In [1]:
import tensorflow as tf
from keras.utils import to_categorical
from keras import layers, models, optimizers, metrics
import datetime


import sys
sys.path.append('..')
from scripts.prepare_data import download_data, preproces_baseline_forest, preproces_without_oversampling, augment_minority_classes
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

import numpy as np

2026-01-11 14:18:20.485305: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-11 14:18:20.517098: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-11 14:18:21.330685: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-11 14:18:21.330685: I tensorflow/core/util/port.cc:153] oneDNN custom operations ar

In [2]:
%load_ext tensorboard

In [2]:
df_base_preproc = download_data()
#X_train, X_test, y_train, y_test = preproces_baseline_forest(df_base_preproc)
X_train, X_test, y_train, y_test = preproces_without_oversampling(df_base_preproc)

In [4]:
time_steps = X_train.shape[1]
features = 1

# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_test = scaler.transform(X_test)

X_train_arr = X_train.values if hasattr(X_train, 'values') else X_train
X_test_arr = X_test.values if hasattr(X_test, 'values') else X_test

X_train_preproc = X_train_arr.reshape((X_train.shape[0], time_steps, features))
X_test_preproc = X_test_arr.reshape((X_test.shape[0], time_steps, features))

#augment
X_train_aug, y_train_aug = augment_minority_classes(X_train_preproc, y_train)
perm = np.random.permutation(len(X_train_aug))
X_train_preproc = X_train_aug[perm]
y_train_aug = y_train_aug[perm]

batch_size = 32

y_train_preproc = to_categorical(y_train_aug)
y_test_preproc = to_categorical(y_test)

train_ds = tf.data.Dataset.from_tensor_slices((X_train_preproc, y_train_preproc)) \
    .shuffle(1000) \
    .batch(batch_size) \
    .prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_test_preproc, y_test_preproc)) \
    .batch(batch_size) \
    .prefetch(tf.data.AUTOTUNE)

In [5]:
class_weights_vals = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights_vals))
print(f"Obliczone wagi klas: {class_weight_dict}")

Obliczone wagi klas: {0: np.float64(0.2861067919275137), 1: np.float64(2.19653125), 2: np.float64(22.44220945083014), 3: np.float64(200.8257142857143)}


In [8]:
deep_rnn = models.Sequential([
    layers.Input(shape=(X_train_preproc.shape[1], X_train_preproc.shape[2])),
    layers.BatchNormalization(),
    layers.LSTM(32, recurrent_dropout=0.3, return_sequences=False),
    layers.Dense(4, activation="softmax")
])

deep_rnn.compile(loss="categorical_crossentropy",
                 optimizer=optimizers.Adam(learning_rate=0.0003),
                 metrics=[metrics.F1Score(average="macro")])

log_dir = "../logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1, update_freq="batch")


history = deep_rnn.fit(train_ds,
                       validation_data=val_ds,
                       epochs=10,
                       callbacks=[tensorboard_callback],
                    #    class_weight=class_weight_dict
                       )

deep_rnn.save("models/lstm_with_dense.keras")

Epoch 1/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 354s 23ms/step - f1_score: 0.7025 - loss: 0.7275 - val_f1_score: 0.5872 - val_loss: 0.2415
Epoch 2/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 354s 23ms/step - f1_score: 0.7025 - loss: 0.7275 - val_f1_score: 0.5872 - val_loss: 0.2415
Epoch 2/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 351s 23ms/step - f1_score: 0.8266 - loss: 0.4907 - val_f1_score: 0.6117 - val_loss: 0.2124
Epoch 3/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 351s 23ms/step - f1_score: 0.8266 - loss: 0.4907 - val_f1_score: 0.6117 - val_loss: 0.2124
Epoch 3/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 349s 23ms/step - f1_score: 0.8446 - loss: 0.4492 - val_f1_score: 0.6134 - val_loss: 0.2118
Epoch 4/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 349s 23ms/step - f1_score: 0.8446 - loss: 0.4492 - val_f1_score: 0.6134 - val_loss: 0.2118
Epoch 4/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 348s 23ms/step - f1_score: 0.8532 - loss: 0.4254 - val_f1_score: 0.6188 - val_loss: 0.2151
Epoch 5/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 348s 23ms/s

In [9]:
deep_rnn = models.Sequential([
    layers.Input(shape=(X_train_preproc.shape[1], X_train_preproc.shape[2])),
    layers.BatchNormalization(),
    layers.LSTM(32, recurrent_dropout=0.3, return_sequences=False),
    layers.Dense(4, activation="softmax")
])

deep_rnn.compile(loss="categorical_crossentropy",
                 optimizer=optimizers.Adam(learning_rate=0.003),
                 metrics=[metrics.F1Score(average="macro")])

log_dir = "../logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1, update_freq="batch")


history = deep_rnn.fit(train_ds,
                       validation_data=val_ds,
                       epochs=10,
                       callbacks=[tensorboard_callback],
                       class_weight=class_weight_dict
                       )

Epoch 1/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 115s 26ms/step - f1_score: 0.1266 - loss: 1.3948 - val_f1_score: 0.0728 - val_loss: 1.4209
Epoch 2/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 109s 25ms/step - f1_score: 0.2573 - loss: 1.1524 - val_f1_score: 0.5059 - val_loss: 0.8200
Epoch 3/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 115s 26ms/step - f1_score: 0.4816 - loss: 0.8121 - val_f1_score: 0.5380 - val_loss: 0.5468
Epoch 4/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 114s 26ms/step - f1_score: 0.5063 - loss: 0.7432 - val_f1_score: 0.5177 - val_loss: 0.7475
Epoch 5/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 113s 26ms/step - f1_score: 0.5147 - loss: 0.7216 - val_f1_score: 0.5395 - val_loss: 0.5907
Epoch 6/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 117s 27ms/step - f1_score: 0.5259 - loss: 0.6988 - val_f1_score: 0.5441 - val_loss: 0.4974
Epoch 7/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 116s 26ms/step - f1_score: 0.5357 - loss: 0.6816 - val_f1_score: 0.5337 - val_loss: 0.4949
Epoch 8/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 115s 26ms/step - f1_score: 

In [10]:
deep_rnn = models.Sequential([
    layers.Input(shape=(X_train_preproc.shape[1], X_train_preproc.shape[2])),
    layers.BatchNormalization(),
    layers.LSTM(32, recurrent_dropout=0.3, return_sequences=False),
    layers.Dense(4, activation="softmax")
])

deep_rnn.compile(loss="categorical_crossentropy",
                 optimizer=optimizers.Adam(learning_rate=0.003),
                 metrics=[metrics.F1Score(average="macro")])

log_dir = "../logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1, update_freq="batch")


history = deep_rnn.fit(train_ds,
                       validation_data=val_ds,
                       epochs=30,
                       callbacks=[tensorboard_callback],
                       class_weight=class_weight_dict
                       )

Epoch 1/30
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 116s 26ms/step - f1_score: 0.1373 - loss: 1.3938 - val_f1_score: 0.0739 - val_loss: 1.4551
Epoch 2/30
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 114s 26ms/step - f1_score: 0.2941 - loss: 1.1366 - val_f1_score: 0.4077 - val_loss: 0.4775
Epoch 3/30
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 112s 25ms/step - f1_score: 0.4317 - loss: 0.8689 - val_f1_score: 0.4530 - val_loss: 0.7489
Epoch 4/30
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 112s 26ms/step - f1_score: 0.4699 - loss: 0.7723 - val_f1_score: 0.4903 - val_loss: 0.6368
Epoch 5/30
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 110s 25ms/step - f1_score: 0.4952 - loss: 0.7387 - val_f1_score: 0.4632 - val_loss: 0.6657
Epoch 6/30
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 114s 26ms/step - f1_score: 0.5147 - loss: 0.7279 - val_f1_score: 0.5259 - val_loss: 0.5911
Epoch 7/30
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 112s 25ms/step - f1_score: 0.5164 - loss: 0.6996 - val_f1_score: 0.5667 - val_loss: 0.4328
Epoch 8/30
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 111s 25ms/step - f1_score: 

In [11]:
deep_rnn = models.Sequential([
    layers.Input(shape=(X_train_preproc.shape[1], X_train_preproc.shape[2])),
    layers.BatchNormalization(),
    layers.LSTM(64, recurrent_dropout=0.3, return_sequences=False),
    layers.Dense(4, activation="softmax")
])

deep_rnn.compile(loss="categorical_crossentropy",
                 optimizer=optimizers.Adam(learning_rate=0.003),
                 metrics=[metrics.F1Score(average="macro")])

log_dir = "../logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1, update_freq="batch")


history = deep_rnn.fit(train_ds,
                       validation_data=val_ds,
                       epochs=10,
                       callbacks=[tensorboard_callback],
                       class_weight=class_weight_dict
                       )

Epoch 1/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 118s 27ms/step - f1_score: 0.1170 - loss: 1.3953 - val_f1_score: 0.1119 - val_loss: 1.5275
Epoch 2/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 118s 27ms/step - f1_score: 0.2602 - loss: 1.1342 - val_f1_score: 0.5025 - val_loss: 0.7601
Epoch 3/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 118s 27ms/step - f1_score: 0.4814 - loss: 0.8030 - val_f1_score: 0.4964 - val_loss: 0.6921
Epoch 4/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 117s 27ms/step - f1_score: 0.5079 - loss: 0.7183 - val_f1_score: 0.5306 - val_loss: 0.4925
Epoch 5/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 118s 27ms/step - f1_score: 0.5238 - loss: 0.6954 - val_f1_score: 0.5549 - val_loss: 0.4771
Epoch 6/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 117s 27ms/step - f1_score: 0.5337 - loss: 0.6685 - val_f1_score: 0.5479 - val_loss: 0.4768
Epoch 7/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 116s 26ms/step - f1_score: 0.5413 - loss: 0.6494 - val_f1_score: 0.5621 - val_loss: 0.3541
Epoch 8/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 118s 27ms/step - f1_score: 

In [12]:
deep_rnn = models.Sequential([
    layers.Input(shape=(X_train_preproc.shape[1], X_train_preproc.shape[2])),
    layers.BatchNormalization(),
    layers.LSTM(128, recurrent_dropout=0.3, return_sequences=False),
    layers.Dense(4, activation="softmax")
])

deep_rnn.compile(loss="categorical_crossentropy",
                 optimizer=optimizers.Adam(learning_rate=0.003),
                 metrics=[metrics.F1Score(average="macro")])

log_dir = "../logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1, update_freq="batch")


history = deep_rnn.fit(train_ds,
                       validation_data=val_ds,
                       epochs=10,
                       callbacks=[tensorboard_callback],
                       class_weight=class_weight_dict
                       )

Epoch 1/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 122s 28ms/step - f1_score: 0.1216 - loss: 1.4280 - val_f1_score: 0.0114 - val_loss: 1.4334
Epoch 2/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 119s 27ms/step - f1_score: 0.2264 - loss: 1.2787 - val_f1_score: 0.3887 - val_loss: 0.8396
Epoch 3/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 119s 27ms/step - f1_score: 0.4777 - loss: 0.8402 - val_f1_score: 0.5159 - val_loss: 0.8824
Epoch 4/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 119s 27ms/step - f1_score: 0.5136 - loss: 0.7377 - val_f1_score: 0.5602 - val_loss: 0.5989
Epoch 5/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 121s 27ms/step - f1_score: 0.5210 - loss: 0.7161 - val_f1_score: 0.5968 - val_loss: 0.4454
Epoch 6/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 121s 28ms/step - f1_score: 0.5325 - loss: 0.6727 - val_f1_score: 0.5866 - val_loss: 0.5884
Epoch 7/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 119s 27ms/step - f1_score: 0.5424 - loss: 0.6447 - val_f1_score: 0.5596 - val_loss: 0.5963
Epoch 8/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 121s 27ms/step - f1_score: 

In [ ]:
deep_rnn = models.Sequential([
    layers.Input(shape=(X_train_preproc.shape[1], X_train_preproc.shape[2])),
    layers.BatchNormalization(),
    layers.LSTM(32, recurrent_dropout=0.3, return_sequences=True),
    layers.LSTM(32, recurrent_dropout=0.3, return_sequences=False),
    layers.Dense(4, activation="softmax")
])

deep_rnn.compile(loss="categorical_crossentropy",
                 optimizer=optimizers.Adam(learning_rate=0.003),
                 metrics=[metrics.F1Score(average="macro")])

log_dir = "../logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1, update_freq="batch")


history = deep_rnn.fit(train_ds,
                       validation_data=val_ds,
                       epochs=10,
                       callbacks=[tensorboard_callback],
                       class_weight=class_weight_dict
                       )

Epoch 1/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 203s 46ms/step - f1_score: 0.1434 - loss: 1.3752 - val_f1_score: 0.1026 - val_loss: 1.3898
Epoch 2/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 201s 46ms/step - f1_score: 0.2523 - loss: 1.2632 - val_f1_score: 0.1417 - val_loss: 2.0792
Epoch 3/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 203s 46ms/step - f1_score: 0.4391 - loss: 0.8866 - val_f1_score: 0.5390 - val_loss: 0.6365
Epoch 4/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 202s 46ms/step - f1_score: 0.5185 - loss: 0.7351 - val_f1_score: 0.5363 - val_loss: 0.7545
Epoch 5/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 203s 46ms/step - f1_score: 0.5229 - loss: 0.7263 - val_f1_score: 0.4980 - val_loss: 0.7390
Epoch 6/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 202s 46ms/step - f1_score: 0.5394 - loss: 0.6866 - val_f1_score: 0.5290 - val_loss: 0.5569
Epoch 7/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 201s 46ms/step - f1_score: 0.5416 - loss: 0.6474 - val_f1_score: 0.5874 - val_loss: 0.4763
Epoch 8/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 202s 46ms/step - f1_score: 

In [14]:
deep_rnn.save("double_lstm.keras")

In [ ]:
# sufit f1 ok. 60%
# TODO: 
# 1. Warstwy gęste po LSTM
# 2. Dwukierunkowe czytanie sygnału - warstwa Bidirectional
# 3. 1D Conv + pooling przed LSTM

In [9]:
deep_rnn = models.Sequential([
    layers.Input(shape=(X_train_preproc.shape[1], X_train_preproc.shape[2])),
    layers.BatchNormalization(),
    layers.LSTM(32, recurrent_dropout=0.3, return_sequences=False),
    layers.Dense(256, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(4, activation="softmax")
])

deep_rnn.compile(loss="categorical_crossentropy",
                 optimizer=optimizers.Adam(learning_rate=0.003),
                 metrics=[metrics.F1Score(average="macro")])

log_dir = "../logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1, update_freq="batch")


history = deep_rnn.fit(train_ds,
                       validation_data=val_ds,
                       epochs=10,
                       callbacks=[tensorboard_callback],
                       class_weight=class_weight_dict
                       )

deep_rnn.save("models/lstm_with_dense.keras")

Epoch 1/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 116s 26ms/step - f1_score: 0.1003 - loss: 1.4262 - val_f1_score: 0.0055 - val_loss: 1.4410
Epoch 2/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 115s 26ms/step - f1_score: 0.1391 - loss: 1.4188 - val_f1_score: 0.0447 - val_loss: 1.5003
Epoch 3/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 115s 26ms/step - f1_score: 0.1718 - loss: 1.3774 - val_f1_score: 0.0055 - val_loss: 1.4446
Epoch 4/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 114s 26ms/step - f1_score: 0.2277 - loss: 1.2872 - val_f1_score: 0.0891 - val_loss: 1.1637
Epoch 5/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 116s 26ms/step - f1_score: 0.3559 - loss: 1.0056 - val_f1_score: 0.2205 - val_loss: 0.9751
Epoch 6/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 114s 26ms/step - f1_score: 0.4086 - loss: 0.8930 - val_f1_score: 0.4736 - val_loss: 0.9096
Epoch 7/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 117s 27ms/step - f1_score: 0.5022 - loss: 0.7729 - val_f1_score: 0.4927 - val_loss: 0.7192
Epoch 8/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 115s 26ms/step - f1_score: 

In [10]:
deep_rnn = models.Sequential([
    layers.Input(shape=(X_train_preproc.shape[1], X_train_preproc.shape[2])),
    layers.BatchNormalization(),
    layers.LSTM(64, recurrent_dropout=0.3, return_sequences=False),
    layers.Dense(512, activation="relu"),
    layers.Dense(256, activation="relu"),
    layers.Dense(4, activation="softmax")
])

deep_rnn.compile(loss="categorical_crossentropy",
                 optimizer=optimizers.Adam(learning_rate=0.003),
                 metrics=[metrics.F1Score(average="macro")])

log_dir = "../logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1, update_freq="batch")


history = deep_rnn.fit(train_ds,
                       validation_data=val_ds,
                       epochs=10,
                       callbacks=[tensorboard_callback],
                       class_weight=class_weight_dict
                       )

deep_rnn.save("models/lstm_with_dense_larger.keras")

Epoch 1/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 116s 26ms/step - f1_score: 0.1094 - loss: 1.4575 - val_f1_score: 0.0055 - val_loss: 1.4244
Epoch 2/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 114s 26ms/step - f1_score: 0.0909 - loss: 1.4106 - val_f1_score: 0.0055 - val_loss: 1.4286
Epoch 3/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 117s 27ms/step - f1_score: 0.0801 - loss: 1.4366 - val_f1_score: 0.0055 - val_loss: 1.3962
Epoch 4/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 115s 26ms/step - f1_score: 0.1041 - loss: 1.4183 - val_f1_score: 0.0055 - val_loss: 1.5456
Epoch 5/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 113s 26ms/step - f1_score: 0.1005 - loss: 1.4060 - val_f1_score: 0.0055 - val_loss: 1.4167
Epoch 6/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 115s 26ms/step - f1_score: 0.2762 - loss: 1.1501 - val_f1_score: 0.2654 - val_loss: 0.8410
Epoch 7/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 115s 26ms/step - f1_score: 0.3635 - loss: 0.9413 - val_f1_score: 0.1231 - val_loss: 1.1081
Epoch 8/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 115s 26ms/step - f1_score: 

In [12]:
deep_rnn = models.Sequential([
    layers.Input(shape=(X_train_preproc.shape[1], X_train_preproc.shape[2])),
    layers.BatchNormalization(),
    layers.LSTM(16, recurrent_dropout=0.3, return_sequences=False),
    layers.Dense(128, activation="relu"),
    layers.Dense(256, activation="relu"),
    layers.Dense(4, activation="softmax")
])

deep_rnn.compile(loss="categorical_crossentropy",
                 optimizer=optimizers.Adam(learning_rate=0.003),
                 metrics=[metrics.F1Score(average="macro")])

log_dir = "../logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1, update_freq="batch")


history = deep_rnn.fit(train_ds,
                       validation_data=val_ds,
                       epochs=10,
                       callbacks=[tensorboard_callback],
                       class_weight=class_weight_dict
                       )

deep_rnn.save("models/lstm_with_dense_longerTrain.keras")

Epoch 1/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 114s 26ms/step - f1_score: 0.0983 - loss: 1.4196 - val_f1_score: 0.0058 - val_loss: 1.4004
Epoch 2/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 113s 26ms/step - f1_score: 0.1204 - loss: 1.4170 - val_f1_score: 0.0055 - val_loss: 1.4488
Epoch 3/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 116s 27ms/step - f1_score: 0.1078 - loss: 1.4155 - val_f1_score: 0.0493 - val_loss: 1.4332
Epoch 4/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 113s 26ms/step - f1_score: 0.2021 - loss: 1.3087 - val_f1_score: 0.1238 - val_loss: 1.5489
Epoch 5/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 112s 26ms/step - f1_score: 0.3249 - loss: 1.1684 - val_f1_score: 0.4481 - val_loss: 0.8743
Epoch 6/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 113s 26ms/step - f1_score: 0.4333 - loss: 0.9680 - val_f1_score: 0.5071 - val_loss: 0.8336
Epoch 7/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 114s 26ms/step - f1_score: 0.4395 - loss: 0.9114 - val_f1_score: 0.3356 - val_loss: 0.9207
Epoch 8/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 115s 26ms/step - f1_score: 

In [13]:
deep_rnn = models.Sequential([
    layers.Input(shape=(X_train_preproc.shape[1], X_train_preproc.shape[2])),
    layers.BatchNormalization(),
    layers.Bidirectional(layers.LSTM(16, recurrent_dropout=0.3, return_sequences=False)),
    layers.Dense(128, activation="relu"),
    layers.Dense(256, activation="relu"),
    layers.Dense(4, activation="softmax")
])

deep_rnn.compile(loss="categorical_crossentropy",
                 optimizer=optimizers.Adam(learning_rate=0.003),
                 metrics=[metrics.F1Score(average="macro")])

log_dir = "../logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1, update_freq="batch")


history = deep_rnn.fit(train_ds,
                       validation_data=val_ds,
                       epochs=10,
                       callbacks=[tensorboard_callback],
                       class_weight=class_weight_dict
                       )

deep_rnn.save("models/lstm_with_dense_longerTrain.keras")

Epoch 1/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 157s 35ms/step - f1_score: 0.3425 - loss: 1.0871 - val_f1_score: 0.1777 - val_loss: 0.9914
Epoch 2/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 151s 34ms/step - f1_score: 0.4092 - loss: 0.9256 - val_f1_score: 0.4820 - val_loss: 0.8018
Epoch 3/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 151s 34ms/step - f1_score: 0.4314 - loss: 0.8861 - val_f1_score: 0.3722 - val_loss: 0.9303
Epoch 4/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 152s 35ms/step - f1_score: 0.4772 - loss: 0.8088 - val_f1_score: 0.3575 - val_loss: 1.0440
Epoch 5/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 149s 34ms/step - f1_score: 0.4934 - loss: 0.7634 - val_f1_score: 0.4563 - val_loss: 0.7592
Epoch 6/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 154s 35ms/step - f1_score: 0.5059 - loss: 0.7447 - val_f1_score: 0.4237 - val_loss: 0.8655
Epoch 7/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 151s 34ms/step - f1_score: 0.5024 - loss: 0.7295 - val_f1_score: 0.5290 - val_loss: 0.6145
Epoch 8/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 151s 34ms/step - f1_score: 

In [9]:
deep_rnn = models.Sequential([
    layers.Input(shape=(X_train_preproc.shape[1], X_train_preproc.shape[2])),
    layers.BatchNormalization(),
    layers.Bidirectional(layers.LSTM(32, recurrent_dropout=0.3, return_sequences=True)),
    layers.BatchNormalization(),
    layers.Bidirectional(layers.LSTM(16, recurrent_dropout=0.3, return_sequences=False)),
    layers.Dense(128, activation="relu"),
    layers.Dense(4, activation="softmax")
])

deep_rnn.compile(loss="categorical_crossentropy",
                 optimizer=optimizers.Adam(learning_rate=0.003),
                 metrics=[metrics.F1Score(average="macro")])

log_dir = "../logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1, update_freq="batch")


history = deep_rnn.fit(train_ds,
                       validation_data=val_ds,
                       epochs=10,
                       callbacks=[tensorboard_callback],
                    #    class_weight=class_weight_dict
                       )

deep_rnn.save("models/aug_double_lstm_bidirectional.keras")

Epoch 1/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 1171s 76ms/step - f1_score: 0.9211 - loss: 0.2300 - val_f1_score: 0.7283 - val_loss: 0.0603
Epoch 2/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 1171s 76ms/step - f1_score: 0.9211 - loss: 0.2300 - val_f1_score: 0.7283 - val_loss: 0.0603
Epoch 2/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 1154s 75ms/step - f1_score: 0.9683 - loss: 0.0995 - val_f1_score: 0.7056 - val_loss: 0.0702
Epoch 3/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 1154s 75ms/step - f1_score: 0.9683 - loss: 0.0995 - val_f1_score: 0.7056 - val_loss: 0.0702
Epoch 3/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 1129s 74ms/step - f1_score: 0.9758 - loss: 0.0762 - val_f1_score: 0.7319 - val_loss: 0.0551
Epoch 4/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 1129s 74ms/step - f1_score: 0.9758 - loss: 0.0762 - val_f1_score: 0.7319 - val_loss: 0.0551
Epoch 4/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 1133s 74ms/step - f1_score: 0.9796 - loss: 0.0643 - val_f1_score: 0.7330 - val_loss: 0.0589
Epoch 5/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 1133

In [10]:
from keras import regularizers

deep_rnn = models.Sequential([
    layers.Input(shape=(X_train_preproc.shape[1], X_train_preproc.shape[2])),
    layers.BatchNormalization(),
    layers.Bidirectional(layers.LSTM(32, recurrent_dropout=0.3, return_sequences=True, kernel_regularizer=regularizers.L2())),
    layers.BatchNormalization(),
    layers.Bidirectional(layers.LSTM(16, recurrent_dropout=0.3, return_sequences=False, kernel_regularizer=regularizers.L2())),
    layers.Dense(128, activation="relu"),
    layers.Dense(4, activation="softmax")
])

deep_rnn.compile(loss="categorical_crossentropy",
                 optimizer=optimizers.Adam(learning_rate=0.003),
                 metrics=[metrics.F1Score(average="macro")])

log_dir = "../logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1, update_freq="batch")
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_f1_score", patience = 3, restore_best_weights=True)


history = deep_rnn.fit(train_ds,
                       validation_data=val_ds,
                       epochs=10,
                       callbacks=[tensorboard_callback, early_stopping],
                    #    class_weight=class_weight_dict
                       )

deep_rnn.save("models/aug_double_lstm_bidirectional_l2.keras")

Epoch 1/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 1175s 76ms/step - f1_score: 0.8860 - loss: 0.3814 - val_f1_score: 0.6918 - val_loss: 0.1460
Epoch 2/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 1175s 76ms/step - f1_score: 0.8860 - loss: 0.3814 - val_f1_score: 0.6918 - val_loss: 0.1460
Epoch 2/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 1159s 75ms/step - f1_score: 0.9399 - loss: 0.2249 - val_f1_score: 0.7115 - val_loss: 0.1277
Epoch 3/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 1159s 75ms/step - f1_score: 0.9399 - loss: 0.2249 - val_f1_score: 0.7115 - val_loss: 0.1277
Epoch 3/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 1146s 75ms/step - f1_score: 0.9519 - loss: 0.1881 - val_f1_score: 0.4090 - val_loss: 0.4729
Epoch 4/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 1146s 75ms/step - f1_score: 0.9519 - loss: 0.1881 - val_f1_score: 0.4090 - val_loss: 0.4729
Epoch 4/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 1111s 72ms/step - f1_score: 0.9578 - loss: 0.1698 - val_f1_score: 0.7219 - val_loss: 0.1342
Epoch 5/10
15355/15355 ━━━━━━━━━━━━━━━━━━━━ 1111